In [ ]:
import re, math
import pandas as pd
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#Configuration
CFG = {
    "publications_path": "/content/publications.csv",
    "published_texts_path": "/content/published_texts.csv",
    "oa_lexicon_path": "/content/OA_Lexicon_eBL.csv",

    # page filters
    "min_page_chars": 250,
    "use_has_akkadian_only": True,

    # transliteration detection
    "min_dict_coverage_line": 0.18,     # line: % tokens matching OA lexicon
    "min_translit_density_page": 0.04,  # page: ratio translit-like lines
    "min_translit_lines_block": 1,      # allow short blocks
    "max_block_lines": 30,              # avoid huge OCR blocks

    # translation detection
    "translation_window_lines": 140,
    "min_translation_chars": 25,
    "use_next_prev_page_for_translation": True,
    "force_english_heuristic": True,    # no langdetect (OCR unstable)

    # TF-IDF linking
    "tfidf_ngram_min": 3,
    "tfidf_ngram_max": 5,
    "min_link_sim": 0.18,               # lower = more recall (tune)
    "max_oare_candidates": 5,

    # sentence alignment filters
    "min_src_chars": 8,
    "min_tgt_chars": 8,
    "min_len_ratio": 0.15,
    "max_len_ratio": 6.0,

    # outputs
    "out_blocks_csv": "/content/extra_block_pairs.csv",
    "out_sent_csv": "/content/extra_train_sentences.csv",
}

#CSV loader
def load_publications(path: str) -> pd.DataFrame:
    pub = pd.read_csv(
        path,
        engine="python",
        on_bad_lines="skip",
        dtype={"pdf_name":"string","page":"Int64","page_text":"string","has_akkadian":"string"},
    )
    pub["page_text"] = pub["page_text"].fillna("").astype(str)
    pub["has_akkadian"] = pub["has_akkadian"].astype(str).str.strip().str.lower().isin(["true","1","yes"])
    pub = pub[pub["page_text"].str.len() >= CFG["min_page_chars"]].copy()
    if CFG["use_has_akkadian_only"]:
        pub = pub[pub["has_akkadian"] == True].copy()
    return pub

def load_published_texts(path: str) -> pd.DataFrame:
    published = pd.read_csv(path)
    published["oare_id"] = published["oare_id"].astype(str).str.strip()
    # prefer cleaned transliteration if present
    if "transliteration" in published.columns:
        published["translit_base"] = published["transliteration"].fillna("").astype(str)
    else:
        published["translit_base"] = published["transliteration_orig"].fillna("").astype(str)
    return published

#OCR normalization
def normalize_ocr_page(text: str) -> str:
    text = str(text).replace("\r", "\n")
    text = text.replace("-\n", "")            # undo hyphenation across linebreak
    lines = [ln.strip() for ln in text.split("\n")]
    lines = [ln for ln in lines if ln]
    return "\n".join(lines)


#Build OA lexicon set (dictionary)

SUBSCRIPT_MAP = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")

def norm_token(tok: str) -> str:
    tok = tok.strip()
    tok = tok.translate(SUBSCRIPT_MAP)
    tok = tok.replace("ḫ","h").replace("Ḫ","H")
    tok = tok.lower()
    tok = re.sub(r"[^a-z0-9šṣṭāēīū\-]", "", tok)
    tok = tok.replace("-", "")
    return tok

def load_oa_lexicon(path: str) -> set:
    lex = pd.read_csv(path)
    # use "norm" if present; else form
    toks = set()
    if "norm" in lex.columns:
        toks.update(lex["norm"].dropna().astype(str).map(lambda x: norm_token(x)))
    if "form" in lex.columns:
        toks.update(lex["form"].dropna().astype(str).map(lambda x: norm_token(x)))
    toks = {t for t in toks if len(t) >= 2}
    return toks

OA_TOKENS = load_oa_lexicon(CFG["oa_lexicon_path"])
print("OA lexicon tokens:", len(OA_TOKENS))

#Transliteration detection using dictionary coverage

RE_SIGN = re.compile(r"\b[A-ZÚŠĜḪṢṬ]{2,}(?:\.[A-ZÚŠĜḪṢṬ]{2,})+\b")  # KÙ.BABBAR etc
RE_HYPHEN = re.compile(r"[A-Za-zšṣṭḫāēīū]+(?:-[A-Za-z0-9šṣṭḫāēīū]+){1,}")
RE_LINE_NO = re.compile(r"^\s*\d+(?:'{0,2})?\s+")
RE_DET = re.compile(r"\{[a-zA-Z]+\}")
RE_GAP = re.compile(r"<gap>")

def dict_coverage(line: str) -> float:
    toks = [norm_token(t) for t in line.split()]
    toks = [t for t in toks if len(t) >= 2]
    if not toks:
        return 0.0
    hit = sum(t in OA_TOKENS for t in toks)
    return hit / len(toks)

def is_translit_line(line: str) -> bool:
    ln = line.strip()
    if len(ln) < 3:
        return False

    score = 0
    cov = dict_coverage(ln)
    if cov >= CFG["min_dict_coverage_line"]:
        score += 3
    if RE_SIGN.search(ln): score += 2
    if RE_HYPHEN.search(ln): score += 2
    if RE_LINE_NO.search(ln): score += 1
    if RE_DET.search(ln): score += 1
    if RE_GAP.search(ln): score += 1

    # reject obvious prose (very light)
    if re.search(r"\b(the|and|with|from|was|were|this|that)\b", ln.lower()):
        score -= 2

    return score >= 2

def page_translit_density(page_text_norm: str) -> float:
    lines = [l.strip() for l in page_text_norm.split("\n") if l.strip()]
    if not lines:
        return 0.0
    return sum(is_translit_line(l) for l in lines) / len(lines)

def extract_translit_blocks(page_text_norm: str):
    lines = [l.strip() for l in page_text_norm.split("\n") if l.strip()]
    blocks = []
    i = 0
    while i < len(lines):
        if is_translit_line(lines[i]):
            start = i
            while i < len(lines) and is_translit_line(lines[i]) and (i - start) < CFG["max_block_lines"]:
                i += 1
            end = i
            if (end - start) >= CFG["min_translit_lines_block"]:
                blocks.append((start, end, " ".join(lines[start:end]).strip()))
        else:
            i += 1
    return blocks, lines

#Translation detection (English heuristic + near-block + next/prev pages)
COMMON_EN = ["the","and","to","of","in","for","with","from","his","her","their","this","that","was","were","paid","silver","shekel","mina"]

def looks_like_english(text: str) -> bool:
    lw = " " + text.lower() + " "
    hit = sum((" " + w + " ") in lw for w in COMMON_EN)
    return hit >= 2

def is_translation_line_en(line: str) -> bool:
    ln = line.strip()
    if len(ln) < 12:
        return False
    if is_translit_line(ln):
        return False
    return looks_like_english(ln)

def extract_translation_near(lines, start_idx, end_idx):
    before = lines[max(0, start_idx-60):start_idx]
    after  = lines[end_idx:end_idx + CFG["translation_window_lines"]]
    cand = before + after
    tgt_lines = [l for l in cand if is_translation_line_en(l)]
    return " ".join(tgt_lines).strip()

#Link transliteration block -> OARE via TF-IDF char ngrams
def clean_for_tfidf(s: str) -> str:
    s = s.replace("ḫ","h").replace("Ḫ","H")
    s = s.translate(SUBSCRIPT_MAP)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_tfidf_index(published: pd.DataFrame):
    corpus = published["translit_base"].fillna("").astype(str).map(clean_for_tfidf).tolist()
    vec = TfidfVectorizer(analyzer="char", ngram_range=(CFG["tfidf_ngram_min"], CFG["tfidf_ngram_max"]))
    X = vec.fit_transform(corpus)
    return vec, X

#Sentence splitting + simple monotonic alignment
def split_src(text: str):
    text = clean_for_tfidf(text)
    parts = re.split(r"\s*[;:]\s+|\s*,\s+|\s+\.\s+", text)
    return [p.strip() for p in parts if len(p.strip()) >= 5]

def split_en(text: str):
    parts = re.split(r"(?<=[.!?;:])\s+", text.strip())
    return [p.strip() for p in parts if len(p.strip()) >= 5]

def align_1to1_by_index(src_sents, tgt_sents):
    # conservative: pair by position up to min length
    k = min(len(src_sents), len(tgt_sents))
    pairs = []
    for i in range(k):
        s, t = src_sents[i].strip(), tgt_sents[i].strip()
        if len(s) < CFG["min_src_chars"] or len(t) < CFG["min_tgt_chars"]:
            continue
        ratio = len(t) / max(1, len(s))
        if ratio < CFG["min_len_ratio"] or ratio > CFG["max_len_ratio"]:
            continue
        pairs.append((s, t))
    return pairs

#Main pipeline
def build_extra_training_data():
    print("Loading files...")
    pub = load_publications(CFG["publications_path"])
    published = load_published_texts(CFG["published_texts_path"])

    print("Normalizing OCR pages...")
    pub["page_text_norm"] = pub["page_text"].astype(str).map(normalize_ocr_page)

    print("Filtering by transliteration density...")
    pub["dens"] = pub["page_text_norm"].map(page_translit_density)
    pub = pub[pub["dens"] >= CFG["min_translit_density_page"]].copy()
    print("Pages after density filter:", len(pub))

    # keyed lookup for next/prev page retrieval
    pub_keyed = pub.set_index(["pdf_name", "page"], drop=False)

    print("Building TF-IDF index over published_texts transliterations...")
    vec, X = build_tfidf_index(published)

    # Debug counters
    cnt_pages = 0
    cnt_pages_with_blocks = 0
    cnt_blocks_total = 0
    cnt_link_ok = 0
    cnt_with_translation = 0
    cnt_english_ok = 0

    block_rows = []
    sent_rows = []

    print("Extract blocks -> link OARE -> find translation (same/next/prev) -> align...")
    for _, r in tqdm(pub.iterrows(), total=len(pub)):
        cnt_pages += 1
        page_norm = r["page_text_norm"]
        blocks, lines = extract_translit_blocks(page_norm)
        if blocks:
            cnt_pages_with_blocks += 1
        cnt_blocks_total += len(blocks)

        if not blocks:
            continue

        for (start, end, src_block) in blocks:
            # TF-IDF link
            q = vec.transform([clean_for_tfidf(src_block)])
            sims = cosine_similarity(q, X).ravel()
            top_idx = sims.argsort()[-CFG["max_oare_candidates"]:][::-1]
            best_i = top_idx[0]
            best_sim = float(sims[best_i])
            if best_sim < CFG["min_link_sim"]:
                continue

            cnt_link_ok += 1
            oare_id = published.iloc[best_i]["oare_id"]

            # translation near block
            tgt_block = extract_translation_near(lines, start, end)

            # try next/prev page in same pdf if empty
            if CFG["use_next_prev_page_for_translation"] and len(tgt_block) < CFG["min_translation_chars"]:
                if not pd.isna(r["page"]):
                    p = int(r["page"])
                    pdf = r["pdf_name"]

                    for delta in [+1, -1, +2]:
                        try:
                            neigh = pub_keyed.loc[(pdf, p + delta), "page_text_norm"]
                            neigh_lines = [l.strip() for l in str(neigh).split("\n") if l.strip()]
                            # search translation anywhere on that page
                            cand = [l for l in neigh_lines if is_translation_line_en(l)]
                            if cand:
                                tgt_block = " ".join(cand).strip()
                                break
                        except KeyError:
                            pass

            if len(tgt_block) < CFG["min_translation_chars"]:
                continue
            cnt_with_translation += 1

            if CFG["force_english_heuristic"] and not looks_like_english(tgt_block):
                continue
            cnt_english_ok += 1

            block_rows.append({
                "pdf_name": r["pdf_name"],
                "page": int(r["page"]) if not pd.isna(r["page"]) else None,
                "oare_id": oare_id,
                "link_sim": best_sim,
                "src_block_raw": src_block,
                "tgt_block_raw": tgt_block,
            })

            # sentence alignment
            src_sents = split_src(src_block)
            tgt_sents = split_en(tgt_block)
            for s, t in align_1to1_by_index(src_sents, tgt_sents):
                sent_rows.append({
                    "pdf_name": r["pdf_name"],
                    "page": int(r["page"]) if not pd.isna(r["page"]) else None,
                    "oare_id": oare_id,
                    "link_sim": best_sim,
                    "src_sentence": s,
                    "tgt_sentence": t,
                })

    blocks_df = pd.DataFrame(block_rows).drop_duplicates()
    sent_df = pd.DataFrame(sent_rows).drop_duplicates()

    print("\n=== DEBUG COUNTERS ===")
    print("Pages processed:", cnt_pages)
    print("Pages with translit blocks:", cnt_pages_with_blocks)
    print("Total blocks found:", cnt_blocks_total)
    print("Blocks passing TF-IDF link:", cnt_link_ok)
    print("Blocks with translation text:", cnt_with_translation)
    print("Blocks passing English heuristic:", cnt_english_ok)

    print("\nSaving outputs...")
    blocks_df.to_csv(CFG["out_blocks_csv"], index=False)
    sent_df.to_csv(CFG["out_sent_csv"], index=False)

    print("Block pairs:", len(blocks_df))
    print("Sentence pairs:", len(sent_df))
    print("Saved:", CFG["out_blocks_csv"])
    print("Saved:", CFG["out_sent_csv"])

    return blocks_df, sent_df

blocks_df, sent_df = build_extra_training_data()

#Show a few samples (if any)
if len(blocks_df) > 0:
    print("\nSample block pair:\n", blocks_df.sample(1, random_state=0).iloc[0][["src_block_raw","tgt_block_raw"]])
if len(sent_df) > 0:
    print("\nSample sentence pairs:")
    print(sent_df.sample(min(10, len(sent_df)), random_state=0)[["src_sentence","tgt_sentence"]].to_string(index=False))

OA lexicon tokens: 47878
Loading files...
Normalizing OCR pages...
Filtering by transliteration density...
Pages after density filter: 20656
Building TF-IDF index over published_texts transliterations...
Extract blocks -> link OARE -> find translation (same/next/prev) -> align...


Streaming output truncated to the last 5000 lines.
 64%|██████▍   | 13205/20656 [46:37<27:49,  4.46it/s]/tmp/ipykernel_173/44452619.py:313: PerformanceWarning: indexing past lexsort depth may impact performance.
  neigh = pub_keyed.loc[(pdf, p + delta), "page_text_norm"]
 64%|██████▍   | 13208/20656 [46:38<25:29,  4.87it/s]/tmp/ipykernel_173/44452619.py:313: PerformanceWarning: indexing past lexsort depth may impact performance.
  neigh = pub_keyed.loc[(pdf, p + delta), "page_text_norm"]
/tmp/ipykernel_173/44452619.py:313: PerformanceWarning: indexing past lexsort depth may impact performance.
  neigh = pub_keyed.loc[(pdf, p + delta), "page_text_norm"]
 64%|██████▍   | 13209/20656 [46:38<25:45,  4.82it/s]/tmp/ipykernel_173/44452619.py:313: PerformanceWarning: indexing past lexsort depth may impact performance.
  neigh = pub_keyed.loc[(pdf, p + delta), "page_text_norm"]
/tmp/ipykernel_173/44452619.py:313: PerformanceWarning: indexing past lexsort depth may impact performance.
  neigh = 


=== DEBUG COUNTERS ===
Pages processed: 20656
Pages with translit blocks: 20656
Total blocks found: 20660
Blocks passing TF-IDF link: 6434
Blocks with translation text: 830
Blocks passing English heuristic: 830

Saving outputs...
Block pairs: 415
Sentence pairs: 968
Saved: /content/extra_block_pairs.csv
Saved: /content/extra_train_sentences.csv

Sample block pair:
 src_block_raw    257\n(4.1) communicates. Another letter, updat...
tgt_block_raw    Stratford, Edward - Agents, Archives, and Risk...
Name: 699, dtype: object

Sample sentence pairs:
                                                                                                                                                                                                                   src_sentence                                                                                          tgt_sentence
                                                                                                                          